In [226]:
import numpy as np
from numpy.linalg import norm
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [227]:
def power_method(A):
    m,n=A.shape
    v=np.random.rand(n).reshape(-1,1)
    B=A.T@A
    for _ in range(500):
        v_old=v
        y=B@v
        if norm(y)<1e-10:
            return v
        v=y/norm(y)
        # Convergence check:
        if norm(v-v_old)<1e-10 or norm(v+v_old)<1e-10:
            break
    return v

def mySVD_reduced(A_input):
    #Assume m>>n.
    A=A_input.copy().astype(float) # Work on a copy
    m, n=A.shape
    U=np.zeros((m,n))
    S=np.zeros((n,n))
    V=np.zeros((n,n))
    
    #We only need to iterate up to the rank of A.
    for j in range(n):
        v=power_method(A)
        V[:,j]=v.flatten()
        sigma=norm(A@v)
        S[j,j]=sigma
        if sigma>1e-10:
            u=(A@v)/sigma
            U[:,j]=u.flatten()
            #Deflation
            A=A-sigma*u@v.T
        else:
            #If sigma is 0, we have reached the rank limit.
            break
            
    return U,S,V

In [228]:
train=pd.read_csv('Train.csv').copy()
test=pd.read_csv('Test.csv').copy()
train=train.drop(columns=['Segmentation'])

#Concatenate
df_all=pd.concat([train, test])
df_all=df_all.drop(columns=['ID'])

#Ordinal encoding
df_all['Gender']=df_all['Gender'].map({'Male':1, 'Female':0})
df_all['Ever_Married']=df_all['Ever_Married'].map({'Yes': 1, 'No': 0})
df_all['Graduated']=df_all['Graduated'].map({'Yes': 1, 'No': 0})
df_all['Spending_Score']=df_all['Spending_Score'].map({'Low': 0, 'Average': 1, 'High': 2})

obj_cols=['Profession', 'Var_1']
num_cols = [col for col in df_all.columns if col not in obj_cols]

#Impute any missing values
my_imputer=SimpleImputer()
df_all[num_cols]=my_imputer.fit_transform(df_all[num_cols])

obj_imputer = SimpleImputer(strategy='most_frequent')
df_all[obj_cols] = obj_imputer.fit_transform(df_all[obj_cols])

#Apply one-hot encoder
encoder=OneHotEncoder(sparse_output=False)
OH_cols=pd.DataFrame(encoder.fit_transform(df_all[obj_cols]))

#Get the descriptive names
OH_cols.columns=encoder.get_feature_names_out(obj_cols)

#One-hot encoding removed index, so put it back
OH_cols.index=df_all.index

#Remove categorical columns (will replace with one-hot encoding)
df=df_all.drop(obj_cols, axis=1)

#Add one-hot encoded columns to numerical features
df=pd.concat([df, OH_cols], axis=1)

#Ensure all columns have string type
df.columns=df.columns.astype(str)
df

,Gender,Ever_Married,Age,Graduated,Work_Experience,Spending_Score,Family_Size,Profession_Artist,Profession_Doctor,Profession_Engineer,...,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
0,1.0,0.0,22.0,0.0,1.000000,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,0.0,1.0,38.0,1.0,2.619777,1.0,3.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,0.0,1.0,67.0,1.0,1.000000,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,1.0,67.0,1.0,0.000000,2.0,2.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.0,40.0,1.0,2.619777,2.0,6.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2622,1.0,0.0,29.0,0.0,9.000000,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2623,0.0,0.0,35.0,1.0,1.000000,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2624,0.0,0.0,53.0,1.0,2.619777,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2625,1.0,1.0,47.0,1.0,1.000000,2.0,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [229]:
#Standardization
scaler=StandardScaler()
df_scaled=pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
df_scaled

,Gender,Ever_Married,Age,Graduated,Work_Experience,Spending_Score,Family_Size,Profession_Artist,Profession_Doctor,Profession_Engineer,...,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
0,0.911604,-1.202108,-1.282499,-1.284070,-0.504312,-0.733119,0.768669,-0.694499,-0.308607,-0.309514,...,-0.177873,-0.292706,-0.19788,-0.125946,-0.235726,-0.3367,2.500169,-0.103798,-1.381529,-0.160627
1,-1.096967,0.846918,-0.328606,0.786273,0.000000,0.618028,0.103701,-0.694499,-0.308607,3.230867,...,-0.177873,-0.292706,-0.19788,-0.125946,-0.235726,-0.3367,2.500169,-0.103798,-1.381529,-0.160627
2,-1.096967,0.846918,1.400325,0.786273,-0.504312,-0.733119,-1.226237,-0.694499,-0.308607,3.230867,...,-0.177873,-0.292706,-0.19788,-0.125946,-0.235726,-0.3367,-0.399973,-0.103798,0.723836,-0.160627
3,0.911604,0.846918,1.400325,0.786273,-0.815659,1.969176,-0.561268,-0.694499,-0.308607,-0.309514,...,-0.177873,3.416402,-0.19788,-0.125946,-0.235726,-0.3367,-0.399973,-0.103798,0.723836,-0.160627
4,-1.096967,0.846918,-0.209369,0.786273,0.000000,1.969176,2.098607,-0.694499,-0.308607,-0.309514,...,-0.177873,-0.292706,-0.19788,-0.125946,-0.235726,-0.3367,-0.399973,-0.103798,0.723836,-0.160627
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10690,0.911604,-1.202108,-0.865170,-1.284070,1.986462,-0.733119,0.768669,-0.694499,-0.308607,-0.309514,...,-0.177873,-0.292706,-0.19788,-0.125946,-0.235726,-0.3367,-0.399973,-0.103798,0.723836,-0.160627
10691,-1.096967,-1.202108,-0.507461,0.786273,-0.504312,-0.733119,-1.226237,-0.694499,3.240370,-0.309514,...,-0.177873,-0.292706,-0.19788,-0.125946,-0.235726,-0.3367,-0.399973,-0.103798,0.723836,-0.160627
10692,-1.096967,-1.202108,0.565669,0.786273,0.000000,-0.733119,-0.561268,-0.694499,-0.308607,-0.309514,...,-0.177873,-0.292706,-0.19788,-0.125946,-0.235726,-0.3367,-0.399973,-0.103798,0.723836,-0.160627
10693,0.911604,0.846918,0.207959,0.786273,-0.504312,1.969176,1.433638,-0.694499,-0.308607,-0.309514,...,-0.177873,-0.292706,-0.19788,-0.125946,-0.235726,-0.3367,2.500169,-0.103798,-1.381529,-0.160627


In [230]:
A=df_scaled.to_numpy()
U,S,V=mySVD_reduced(A)
S_list=[S[j,j] for j in range(S.shape[1])]
print(len(S_list))
S_retained=S_list[:15]
print(S_list)
print('Ratio of Variance Retained:',sum([sigma**2 for sigma in S_retained])/sum([sigma**2 for sigma in S_list]))

23
[180.37975179414482, 142.22667453756577, 129.39633383975465, 123.24983775656771, 116.70945767251861, 115.64564414915836, 111.77505634102847, 109.17871149667216, 108.46026335167869, 105.40760487976102, 104.48533324179495, 103.8130727617967, 103.72667601057238, 102.70503061511063, 94.6816582369022, 91.85695519941159, 85.81245772617555, 82.71950668475527, 76.55376301341249, 61.50393174991331, 52.43106582497049, 1.0201552073414023e-13, 0.0]
Ratio of Variance Retained: 0.8575676410291818


The singular values represent the importance of each pattern, i.e. how much each pattern dominates the overall dataset.

If $\sigma_k$ is very small, the patterns in $u_k$ and $v_k$ are just "noise".

By keeping the top 15 singular values, I captured 85.76% of the variance in customer behavior.

In [232]:
#Analyze the columns of V
for j in range(3):
    arr=V[:,j]
    sort=sorted(arr)
    sort_reversed=sorted(arr, reverse=True)
    index_sort=np.argsort(arr)
    index_sort_reversed=np.argsort(-arr)
    col_sort=[df.columns[j] for j in index_sort[:5]]
    col_sort_reversed=[df.columns[j] for j in index_sort_reversed[:5]]
    
    print(f'At column {j}: (0-based indexing)')
    print('Largest values in descending order:')
    print('columns:', col_sort_reversed, 'values:', np.round(sort_reversed[:5],3))
    print('Smallest values in ascending order:')
    print('columns:', col_sort, 'values:', np.round(sort[:5],3))
    print()

At column 0: (0-based indexing)
Largest values in descending order:
columns: ['Age', 'Ever_Married', 'Spending_Score', 'Var_1_Cat_6', 'Profession_Lawyer'] values: [0.468 0.42  0.351 0.265 0.261]
Smallest values in ascending order:
columns: ['Profession_Healthcare', 'Family_Size', 'Var_1_Cat_4', 'Var_1_Cat_3', 'Var_1_Cat_2'] values: [-0.342 -0.202 -0.158 -0.105 -0.104]

At column 1: (0-based indexing)
Largest values in descending order:
columns: ['Var_1_Cat_4', 'Spending_Score', 'Profession_Executive', 'Family_Size', 'Ever_Married'] values: [0.364 0.334 0.331 0.315 0.252]
Smallest values in ascending order:
columns: ['Var_1_Cat_6', 'Profession_Artist', 'Graduated', 'Work_Experience', 'Profession_Homemaker'] values: [-0.387 -0.326 -0.3   -0.189 -0.076]

At column 2: (0-based indexing)
Largest values in descending order:
columns: ['Profession_Engineer', 'Var_1_Cat_3', 'Profession_Artist', 'Var_1_Cat_4', 'Graduated'] values: [0.301 0.298 0.246 0.23  0.17 ]
Smallest values in ascending orde

Matrix $V$: The "Feature Profiles", which tell us what the stereotypical behaviors are.

The right-singular vectors $v_j$ have an entry for every feature (Age, Profession, etc.).

It shows how each feature contributes to a specific archetype.

The first column of $V$: 

The Positive Profile: This dimension is most strongly defined by higher Age, being Married, and having a higher Spending Score. The high loading for the Lawyer profession suggests a segment of high-earning individuals.

The Negative Profile: We find Healthcare workers and customers with larger Family Sizes.

The second column of $V$:

The Positive Profile: This dimension is driven by Executives with large Family Sizes who are Married and have a high Spending Score. The inclusion of Category 4 suggests a specific category that aligns with these households.

The Negative Profile: We see a strong push from Artists, those who have Graduated, and those with more Work Experience. Category 6 is strongly negative here.


In [234]:
#Analyze the columns of U
for j in range(3):
    arr=U[:,j]
    sort=sorted(arr)
    sort_reversed=sorted(arr, reverse=True)
    index_sort=np.argsort(arr)
    index_sort_reversed=np.argsort(-arr)
    
    print(f'At column {j}: (0-based indexing)')
    print('Largest values in descending order:')
    print('indexes:', index_sort_reversed[:5], 'values:', np.round(sort_reversed[:5],3))
    print('Smallest values in ascending order:')
    print('indexes:', index_sort[:5], 'values:', np.round(sort[:5],3))
    print()

At column 0: (0-based indexing)
Largest values in descending order:
indexes: [4301 6855 5651 9577 7102] values: [0.022 0.022 0.022 0.022 0.022]
Smallest values in ascending order:
indexes: [6098 6303 9529 6370 5512] values: [-0.024 -0.023 -0.023 -0.023 -0.023]

At column 1: (0-based indexing)
Largest values in descending order:
indexes: [9040 4480 3040 5558 9027] values: [0.038 0.038 0.037 0.036 0.036]
Smallest values in ascending order:
indexes: [10672  7726  7798  2702  3177] values: [-0.023 -0.023 -0.022 -0.022 -0.022]

At column 2: (0-based indexing)
Largest values in descending order:
indexes: [  343 10008  4520  9356  2115] values: [0.026 0.026 0.026 0.026 0.026]
Smallest values in ascending order:
indexes: [6879  940 1447 5762 7900] values: [-0.025 -0.023 -0.023 -0.023 -0.023]



In [235]:
df.iloc[[4301, 6855]]

,Gender,Ever_Married,Age,Graduated,Work_Experience,Spending_Score,Family_Size,Profession_Artist,Profession_Doctor,Profession_Engineer,...,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
4301,1.0,1.0,89.0,1.0,0.0,2.0,2.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
6855,1.0,1.0,88.0,1.0,0.0,2.0,2.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [236]:
df.iloc[[6098, 6303]] 

,Gender,Ever_Married,Age,Graduated,Work_Experience,Spending_Score,Family_Size,Profession_Artist,Profession_Doctor,Profession_Engineer,...,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
6098,0.0,0.0,19.0,0.0,9.0,0.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
6303,0.0,0.0,22.0,0.0,1.0,0.0,9.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [237]:
df.iloc[[9040, 10672]]

,Gender,Ever_Married,Age,Graduated,Work_Experience,Spending_Score,Family_Size,Profession_Artist,Profession_Doctor,Profession_Engineer,...,Profession_Homemaker,Profession_Lawyer,Profession_Marketing,Var_1_Cat_1,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7
972,1.0,1.0,55.0,0.0,0.0,2.0,8.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2604,0.0,0.0,31.0,1.0,14.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Matrix $U$ is the "Customer Archetypes" that tells us who is the most like the stereotypical behaviors.

Each left-singular vector $u_j$ is a column with an entry for every customer. 

Each column groups customers together who have similar behavior patterns.

The first column of $U$:

The Archetypes for Positive Values: Customers like #4301 and #6855 have higher age, being married, and having a higher Spending Score.

The Archetypes for Negative Values: Customers like #6098 and #6303 are the opposites. They represent the "Healthcare/Large Family" stereotype.

The second column of $U$:

The Archetypes for Positive Values: Customer #9040 is the strongest example of large Family Sizes who are married and have a high Spending Score.

The Archetypes for Negative Values: Customer #10672 is the most extreme example of the Artists, those who have Graduated, and those with more Work Experience.